# RC9.2.2 RC5 — persistent Colab runner

This is the canonical Drive-backed runner for the RC5 validation package. It uses `rc922_runner.py`, stores checkpoints/results in Drive, and supports safe interruption and exact resume. Keep RC9.1 live until the required release gates pass.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import pathlib, zipfile, shutil, json, hashlib, os, sys, subprocess, signal, multiprocessing, time
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/RC922_RC5')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent results root:', DRIVE_ROOT)

In [ ]:
# Upload the exact RC5 validation ZIP. Do not use an RC2/RC3/RC4 package here.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
if not zip_names:
    raise RuntimeError('Upload RC9_2_2_FIX_VALIDATION_RC5.zip')
PACKAGE_ZIP = pathlib.Path('/content') / zip_names[0]
PACKAGE_ROOT = pathlib.Path('/content/rc922_rc4_package')
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
with zipfile.ZipFile(PACKAGE_ZIP) as z:
    z.extractall('/content')
roots = sorted(pathlib.Path('/content').glob('RC9_2_2_FIX_VALIDATION_RC5'))
if roots:
    PACKAGE_ROOT = roots[0]
else:
    candidates = sorted(pathlib.Path('/content').glob('*/SCENARIOS.json'))
    if not candidates:
        raise RuntimeError('The ZIP does not contain SCENARIOS.json')
    PACKAGE_ROOT = candidates[0].parent
manifest = json.loads((PACKAGE_ROOT / 'SCENARIOS.json').read_text())
print('Package:', manifest['package'])
print('Engine :', manifest['engine_release'])
print('SHA256 :', manifest['engine_sha256'])

In [ ]:
# Upload the workbook only when testing an ad-hoc workbook. Leave blank to use a manifest scenario.
WORKBOOK = ''  #@param {type:'string'}
if WORKBOOK.strip():
    if not pathlib.Path(WORKBOOK).is_file():
        print('Upload the workbook now')
        files.upload()
    workbook_path = pathlib.Path(WORKBOOK)
else:
    workbook_path = None
RUN_ID = 'RC5_RUN_01'  #@param {type:'string'}
MODE = 'QUICK'  #@param ['SMOKE', 'QUICK', 'DEEP', 'OVERNIGHT']
STAGE = 'FULL_SCHEDULE'  #@param ['FULL_SCHEDULE', 'BEFORE_BREAKS_ONLY']
ONLY = ''  #@param {type:'string'}
LANGUAGE_WORKING_WINDOW = 'workbook'  #@param ['workbook', 'OFF', 'MINIMUM_ROWS', 'ALL_ROWS', 'REQUIRED_LANGUAGE_ONLY']
NUM_WORKERS = min(2, multiprocessing.cpu_count())  #@param {type:'integer'}
RESUME = True  #@param {type:'boolean'}
RESULTS_ROOT = DRIVE_ROOT / RUN_ID
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Results persist at:', RESULTS_ROOT)

In [ ]:
# Run the canonical RC5 runner. Interrupting this cell asks the runner to stop
# its entire solver process tree; resume can then continue from Drive checkpoints.
cmd = [sys.executable, '-u', str(PACKAGE_ROOT / 'runners' / 'rc922_runner.py'),
       '--package-root', str(PACKAGE_ROOT), '--results-root', str(RESULTS_ROOT),
       '--shard', '0', '--shards', '1', '--mode', MODE, '--stage', STAGE,
       '--num-workers', str(NUM_WORKERS)]
if ONLY.strip(): cmd += ['--only', ONLY.strip()]
if workbook_path is not None: cmd += ['--input', str(workbook_path)]
if LANGUAGE_WORKING_WINDOW != 'workbook': cmd += ['--language-working-window', LANGUAGE_WORKING_WINDOW]
if RESUME: cmd += ['--resume']
print(' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=(os.name == 'posix'))
try:
    for line in proc.stdout:
        print(line, end='')
    print('runner exit code:', proc.wait())
except KeyboardInterrupt:
    print('Interrupt received; stopping the runner safely. Checkpoints remain in Drive.')
    try:
        proc.send_signal(signal.SIGINT)
        print('runner exit code:', proc.wait(timeout=30))
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        else: proc.terminate()
        proc.wait(timeout=30)
    raise

In [ ]:
# Re-score existing results without spending solver time.
cmd = [sys.executable, str(PACKAGE_ROOT / 'runners' / 'rc922_runner.py'),
       '--package-root', str(PACKAGE_ROOT), '--results-root', str(RESULTS_ROOT), '--gate-only']
print(subprocess.run(cmd, text=True, capture_output=True).stdout)

In [ ]:
# Create a persistent download archive after the run or after resume completes.
archive = shutil.make_archive(str(DRIVE_ROOT / (RUN_ID + '_RESULTS')), 'zip', root_dir=RESULTS_ROOT)
print('Download later from Drive:', archive)
print('Do not approve a schedule unless the case has zero return code, independent validation PASS, matching hashes, and the sealed production artifacts.')